In [99]:
%pip install kaggle
%pip install kagglehub
import os
import kagglehub
os.environ["KAGGLE_API_TOKEN"] = "KGAT_67c82fb30c76d3592439bb44a9504e0f"



In [100]:
from pathlib import Path
path = Path(kagglehub.competition_download("us-patent-phrase-to-phrase-matching"))
print(path)


/root/.cache/kagglehub/competitions/us-patent-phrase-to-phrase-matching


In [101]:
import pandas as pd


In [102]:
print(path)

/root/.cache/kagglehub/competitions/us-patent-phrase-to-phrase-matching


In [103]:
print(list(path.iterdir()))

[PosixPath('/root/.cache/kagglehub/competitions/us-patent-phrase-to-phrase-matching/test.csv'), PosixPath('/root/.cache/kagglehub/competitions/us-patent-phrase-to-phrase-matching/train.csv'), PosixPath('/root/.cache/kagglehub/competitions/us-patent-phrase-to-phrase-matching/sample_submission.csv')]


In [104]:
!ls {path}


sample_submission.csv  test.csv  train.csv


In [105]:
df = pd.read_csv(path/'train.csv')
df.head()

,id,anchor,target,context,score
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75
2,36d72442aefd8232,abatement,active catalyst,A47,0.25
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00


In [106]:
df.describe()
df.info()
df.isnull().sum()
df.describe(include='object')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36473 entries, 0 to 36472
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       36473 non-null  object 
 1   anchor   36473 non-null  object 
 2   target   36473 non-null  object 
 3   context  36473 non-null  object 
 4   score    36473 non-null  float64
dtypes: float64(1), object(4)
memory usage: 1.4+ MB


,id,anchor,target,context
count,36473,36473,36473,36473
unique,36473,733,29340,106
top,8d135da0b55b8c88,component composite coating,composition,H01
freq,1,152,24,2186


In [107]:
df['input'] = 'TEXT1: ' + df.context + '; TEXT2: ' + df.target + '; ANC1: ' + df.anchor

In [108]:
df.input.head()

,input
0,TEXT1: A47; TEXT2: abatement of pollution; ANC...
1,TEXT1: A47; TEXT2: act of abating; ANC1: abate...
2,TEXT1: A47; TEXT2: active catalyst; ANC1: abat...
3,TEXT1: A47; TEXT2: eliminating process; ANC1: ...
4,TEXT1: A47; TEXT2: forest region; ANC1: abatement


In [109]:
df.head()

,id,anchor,target,context,score,input
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50,TEXT1: A47; TEXT2: abatement of pollution; ANC...
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75,TEXT1: A47; TEXT2: act of abating; ANC1: abate...
2,36d72442aefd8232,abatement,active catalyst,A47,0.25,TEXT1: A47; TEXT2: active catalyst; ANC1: abat...
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50,TEXT1: A47; TEXT2: eliminating process; ANC1: ...
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00,TEXT1: A47; TEXT2: forest region; ANC1: abatement


In [110]:
from datasets import Dataset, DatasetDict
ds = Dataset.from_pandas(df)  

In [111]:
ds

Dataset({
    features: ['id', 'anchor', 'target', 'context', 'score', 'input'],
    num_rows: 36473
})

In [112]:
model_nm = 'microsoft/deberta-v3-small'

In [113]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokz=AutoTokenizer.from_pretrained(model_nm)

In [114]:
def tok_func(x):
    return tokz(x["input"], truncation=True, max_length=128)

tok_ds = ds.map(tok_func, batched = True)    

Map:   0%|          | 0/36473 [00:00<?, ? examples/s]

In [115]:
row = tok_ds[0]
row['input'], row['input_ids']

('TEXT1: A47; TEXT2: abatement of pollution; ANC1: abatement',
 [54453,
  435,
  294,
  336,
  5753,
  346,
  54453,
  445,
  294,
  47284,
  265,
  6435,
  346,
  23702,
  435,
  294,
  47284])

In [116]:
tokz.vocab['▁of']

265

In [117]:
df.head()

,id,anchor,target,context,score,input
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50,TEXT1: A47; TEXT2: abatement of pollution; ANC...
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75,TEXT1: A47; TEXT2: act of abating; ANC1: abate...
2,36d72442aefd8232,abatement,active catalyst,A47,0.25,TEXT1: A47; TEXT2: active catalyst; ANC1: abat...
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50,TEXT1: A47; TEXT2: eliminating process; ANC1: ...
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00,TEXT1: A47; TEXT2: forest region; ANC1: abatement


In [118]:
tok_ds = tok_ds.rename_column('score','labels')


In [119]:
!ls {path}

sample_submission.csv  test.csv  train.csv


In [120]:
eval_df = pd.read_csv(path/'test.csv')
eval_df.head()
eval_df.info()
eval_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       36 non-null     object
 1   anchor   36 non-null     object
 2   target   36 non-null     object
 3   context  36 non-null     object
dtypes: object(4)
memory usage: 1.3+ KB


,id,anchor,target,context
count,36,36,36,36
unique,36,34,36,29
top,4112d61851461f60,hybrid bearing,inorganic photoconductor drum,G02
freq,1,2,1,3


In [121]:
eval_df['input'] = 'TEXT1: ' + eval_df.context + '; TEXT2: ' + eval_df.target + '; ANC1: ' + eval_df.anchor
eval_ds = Dataset.from_pandas(eval_df).map(tok_func, batched=True)

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

In [130]:
from transformers import TrainingArguments, Trainer
bs = 8
epochs = 2
lr = 2e-6

In [123]:
df = df.dropna(subset=['anchor', 'target', 'context', 'score']).copy()
df['score'] = df['score'].astype('float32')
df['input'] = 'TEXT1: ' + df.context + '; TEXT2: ' + df.target + '; ANC1: ' + df.anchor

ds = Dataset.from_pandas(df)
tok_ds = ds.map(tok_func, batched=True)
tok_ds = tok_ds.rename_column('score', 'labels')


Map:   0%|          | 0/36473 [00:00<?, ? examples/s]

In [124]:
import numpy as np

def corr_d(eval_pred):
    preds, labels = eval_pred
    preds = np.squeeze(preds)

    mask = np.isfinite(preds) & np.isfinite(labels)
    preds, labels = preds[mask], labels[mask]

    if len(preds) < 2 or np.std(preds) == 0 or np.std(labels) == 0:
        return {'pearson': 0.0}

    return {'pearson': float(np.corrcoef(preds, labels)[0,1])}


In [135]:
dds = tok_ds.train_test_split(0.25, seed=42)

args = TrainingArguments(
    'outputs',
    learning_rate=lr,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=False,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs,
    weight_decay=0.01,
    max_grad_norm=0.5,
    report_to='none',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='pearson',
    greater_is_better=True
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [136]:
import numpy as np
import torch

model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=1)
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
trainer = Trainer(model, args, train_dataset=dds['train'], eval_dataset=dds['test'],
                  processing_class=tokz, compute_metrics=corr_d)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight     

In [137]:
import torch
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=1)
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()

sample = dds['test'][:8]

features = [
    {
        'input_ids': sample['input_ids'][i],
        'attention_mask': sample['attention_mask'][i]
    }
    for i in range(len(sample['input_ids']))
]

batch = tokz.pad(features, padding=True, return_tensors='pt')
batch = {k: v.to(model.device) for k, v in batch.items()}
labels = torch.tensor(sample['labels'], dtype=torch.float32).to(model.device)

with torch.no_grad():
    out = model(**batch, labels=labels)

print('loss:', out.loss)
print('nan in logits:', torch.isnan(out.logits).any().item())
print('nan in labels:', torch.isnan(labels).any().item())
print('logits sample:', out.logits[:5])


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight     

loss: tensor(0.0613, device='cuda:0')
nan in logits: False
nan in labels: False
logits sample: tensor([0.4851, 0.3982, 0.4468, 0.3708, 0.3574], device='cuda:0')


In [138]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Pearson
1,0.076450,0.067095,0.006009
2,0.067942,0.065521,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

TrainOutput(global_step=6840, training_loss=0.07219609266136125, metrics={'train_runtime': 467.6707, 'train_samples_per_second': 116.98, 'train_steps_per_second': 14.626, 'total_flos': 280225277569200.0, 'train_loss': 0.07219609266136125, 'epoch': 2.0})

In [139]:
import numpy as np

labels = np.array(dds['test']['labels'], dtype=np.float32)
print("num labels:", len(labels))
print("nan labels:", np.isnan(labels).sum())
print("min/max:", labels.min(), labels.max())
print("unique sample:", np.unique(labels)[:10])


num labels: 9119
nan labels: 0
min/max: 0.0 1.0
unique sample: [0.   0.25 0.5  0.75 1.  ]


In [140]:
preds = trainer.predict(eval_ds).predictions.astype(float)
preds

array([[0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719],
       [0.40136719]])